In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [4]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [5]:
anime_data = anime_data_client.get_cache()

Build features

In [6]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.41006426159563164)

Convert each anime in df to vectors

In [7]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [8]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tuning Bayesian

In [19]:
weights_uncertainty = np.array([
    0, 1, 2, 3,
    3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8,
    8.5, 9, 10, 11, 12
])
n_runs = 1000
tuning_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator

hitman = HitRateEvaluator(
            anime_df_scaled=anime_df_scaled,
            anime_df=anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = hitman.tune_bayesian_uncertainty(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
)

best_bayesian_weights = best_bayesian_weights.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics = bayesian_summary.merge(
    baseline_summary,
    on="k",
    how="left",
).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})

best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,8.5,5,0.6474,0.213726,0.104419,0.034472,3.237,0.0984,0.122689,0.015871,0.019788,0.492
1,10.0,10,0.4155,0.138669,0.134032,0.044732,4.155,0.0643,0.059658,0.020742,0.019244,0.643


In [20]:
bayesian_summary

,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
28,8.5,5,0.6474,0.213726,0.104419,0.034472,3.237
26,8.0,5,0.6460,0.216913,0.104194,0.034986,3.230
30,9.0,5,0.6436,0.212189,0.103806,0.034224,3.218
24,7.5,5,0.6424,0.219842,0.103613,0.035458,3.212
32,10.0,5,0.6376,0.211639,0.102839,0.034135,3.188
22,7.0,5,0.6320,0.227129,0.101935,0.036634,3.160
34,11.0,5,0.6266,0.209419,0.101065,0.033777,3.133
20,6.5,5,0.6194,0.228289,0.099903,0.036821,3.097
36,12.0,5,0.6160,0.211539,0.099355,0.034119,3.080
18,6.0,5,0.6002,0.223450,0.096806,0.036040,3.001


In [9]:
focused_weights_uncertainty = np.array([8.5, 8, 9, 7.5, 10, 7])
focused_n_runs = 500
focused_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

focused_hitman = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

(
    focused_bayesian_results,
    focused_bayesian_summary,
    focused_best_bayesian_weights,
    focused_baseline_results,
    focused_baseline_summary,
) = focused_hitman.tune_bayesian_uncertainty(
    weights=focused_weights_uncertainty,
    n_runs=focused_n_runs,
    top_ks=focused_top_ks,
    random_state=42,
)

focused_ranking_evaluator = RankingMetricEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

focused_ranking_results, focused_ranking_summary = (
    focused_ranking_evaluator.tune_bayesian_uncertainty_ranking(
        weights=focused_weights_uncertainty,
        n_runs=focused_n_runs,
        top_ks=focused_top_ks,
        random_state=42,
    )
)

focused_bayesian_summary = focused_bayesian_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)
focused_ranking_summary = focused_ranking_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

focused_average_metrics = (
    focused_bayesian_summary
    .merge(
        focused_ranking_summary,
        on=["bayesian_uncertainty_weight", "k"],
        how="left",
    )
    .merge(
        focused_baseline_summary,
        on="k",
        how="left",
    )
)

focused_metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / f"bayesian_uncertainty_focused_ndcg_{focused_n_runs}run_20260615.csv"
)
focused_average_metrics.to_csv(focused_metrics_path, index=False)

focused_best_bayesian_weights = (
    focused_average_metrics
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k", "avg_hit_rate"],
        ascending=[True, False, False, False],
    )
    .groupby("k")
    .head(1)
)

focused_best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,8.5,5,0.6604,0.216993,0.103188,0.033905,3.302,0.702652,0.190396,0.987667,0.094453,3.76,2.992,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
6,10.0,10,0.4292,0.138078,0.134125,0.043149,4.292,0.536482,0.144003,0.985952,0.091955,5.28,3.988,46.326,31.87,0.0634,0.059054,0.019813,0.018454,0.634


In [10]:
focused_average_metrics

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,8.5,5,0.6604,0.216993,0.103188,0.033905,3.302,0.702652,0.190396,0.987667,0.094453,3.760,2.992,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
1,8.0,5,0.6560,0.223795,0.102500,0.034968,3.280,0.702498,0.193418,0.985833,0.101929,3.738,2.988,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
2,9.0,5,0.6560,0.213530,0.102500,0.033364,3.280,0.700831,0.188577,0.987333,0.092923,3.768,2.982,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
3,7.5,5,0.6520,0.232129,0.101875,0.036270,3.260,0.699462,0.195837,0.983733,0.110986,3.698,2.982,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
4,10.0,5,0.6484,0.214451,0.101312,0.033508,3.242,0.695568,0.187722,0.985333,0.097887,3.752,2.954,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
5,7.0,5,0.6436,0.238102,0.100562,0.037203,3.218,0.693811,0.199229,0.982633,0.113741,3.634,2.952,46.326,31.87,0.0976,0.121093,0.015250,0.018921,0.488
6,10.0,10,0.4292,0.138078,0.134125,0.043149,4.292,0.536482,0.144003,0.985952,0.091955,5.280,3.988,46.326,31.87,0.0634,0.059054,0.019813,0.018454,0.634
7,9.0,10,0.4228,0.143112,0.132125,0.044723,4.228,0.538222,0.145554,0.987841,0.087721,5.288,3.980,46.326,31.87,0.0634,0.059054,0.019813,0.018454,0.634
8,8.5,10,0.4166,0.147336,0.130188,0.046042,4.166,0.538848,0.147319,0.988175,0.089338,5.286,3.972,46.326,31.87,0.0634,0.059054,0.019813,0.018454,0.634
9,8.0,10,0.4144,0.148583,0.129500,0.046432,4.144,0.538155,0.148676,0.986319,0.097420,5.264,3.954,46.326,31.87,0.0634,0.059054,0.019813,0.018454,0.634


## Results

The initial **1000-run** sweep for user `chekkit` used the selected `mean + popularity + watching + genres + synopsis SVD` feature set and tested uncertainty weights from `0` to `12`. That broad sweep showed that low weights through about `5.5` underperform, while the strongest region is around `7` to `10`. The broad-sweep metrics are mirrored in `metrics/bayesian_uncertainty_tuning_20260615_153918.csv`, and the focused NDCG rerun is saved to `metrics/bayesian_uncertainty_focused_ndcg_500run_20260615.csv`.

| Run | k | Best uncertainty weight | Avg precision@k | Avg hit rate | Avg hits | Avg NDCG@k | Baseline precision@k |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Broad sweep | 5 | 8.5 | 0.6474 | 0.1044 | 3.237 | - | 0.0984 |
| Broad sweep | 10 | 10.0 | 0.4155 | 0.1340 | 4.155 | - | 0.0643 |
| Focused rerun | 5 | 8.5 | 0.6604 | 0.1032 | 3.302 | 0.7027 | 0.0976 |
| Focused rerun | 10 | 10.0 | 0.4292 | 0.1341 | 4.292 | 0.5365 | 0.0634 |

The focused rerun tested the close candidates `[7, 7.5, 8, 8.5, 9, 10]` and added graded NDCG using the ranking-evaluation split. For `k=5`, **8.5** remained the best precision setting and also had the strongest focused NDCG. For `k=10`, **10.0** had the best precision, but the focused NDCG values were neck-and-neck, with **8.5** slightly ahead on NDCG (`0.5388`) despite lower precision than `10.0`.

Conclusion: keep **`bayesian_uncertainty_weight=8.5`** as the default. It is consistently competitive, wins the top-5 cutoff that matters most for short recommendation lists, and sits in the center of the strongest focused range. Use **10.0** only if optimizing specifically for longer top-10 precision.
